In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

In [2]:
SRC=Path.cwd().absolute(); TASK=SRC.parent; ROOT=TASK.parent
INPUT=TASK/'input'; OUTPUT=TASK/'output'
CFG=json.loads((ROOT/'config.json').read_text()); SEED=CFG['seed']
np.random.seed(SEED); torch.manual_seed(SEED)

In [3]:
from fit_models import grid, fit_one, finalists

In [4]:
arrays=np.load(INPUT/'prepare'/'windows.npz')

In [ ]:
screen=pd.DataFrame([fit_one(CFG, arrays, name, 0, 'screen', OUTPUT)
                     for name in grid(CFG)])
screen.to_csv(OUTPUT/'screen_validation.csv', index=False)
display(screen.sort_values('val_nll').head(15))

In [ ]:
selected=finalists(CFG, screen)
(OUTPUT/'selection.json').write_text(json.dumps(selected, indent=2))
display(pd.DataFrame({'selected_config':selected}))

In [ ]:
final=pd.DataFrame([fit_one(CFG, arrays, name, rep, 'final', OUTPUT)
                    for name in selected for rep in range(CFG['fit']['repeats'])])
final.to_csv(OUTPUT/'final_validation.csv', index=False)
display(final.groupby('config')[['val_nll','params']].agg(['mean','std']).sort_values(('val_nll','mean')))